In [21]:
import random

suits = ['Heart', 'Diamond', 'Clubs', 'Spades']
ranks = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'Jack', 'Queen', 'King', 'Ace']

deck = [(rank,suit) for suit in suits for rank in ranks]

face_cards = [card for card in deck if card[0] in ['Jack', 'Queen', 'King', 'Ace']]

trials = 100000
count = 0

for _ in range(trials):
    card = random.choice(face_cards)
    if card[1] == 'Diamond' or card[1] == 'Heart':
        count += 1

simulated_prob = count / trials

print(f"Simulated Probabilty: {simulated_prob:.3f}")

Simulated Probabilty: 0.501


In [25]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Intelligence', 'Grade'),
    ('StudyHours', 'Grade'),
    ('Difficulty', 'Grade'),
    ('Grade', 'Pass'),
])

cpd_intelligence = TabularCPD(variable='Intelligence', variable_card=2, 
    values=[
        [0.7],   # Row 0 = P(Intelligence = High)
        [0.3]    # Row 1 = P(Intelligence = Low)
    ]
)
cpd_studyHours = TabularCPD(variable='StudyHours', variable_card=2, 
    values=[
        [0.6],   # Row 0 = P(StudyHours = Sufficient)
        [0.4]    # Row 1 = P(StudyHours = Insufficient)
    ]
)
cpd_difficulty = TabularCPD(variable='Difficulty', variable_card=2, 
    values=[
        [0.4],   # Row 0 = P(Difficulty = Hard)
        [0.6]    # Row 1 = P(Difficulty = Easy)
    ]
)
cpd_alarm = TabularCPD(variable='Alarm', variable_card=2,
    values=[
        # B=F,E=F  B=F,E=T  B=T,E=F  B=T,E=T
        [0.999,    0.71,    0.06,    0.05],   # Row 0 = Alarm=False
        [0.001,    0.29,    0.94,    0.95]    # Row 1 = Alarm=True
    ],
    evidence=['Burglary', 'Earthquake'],
    evidence_card=[2, 2]
)
# P(Grade | Intelligence, StudyHours, Difficulty)
cpd_grade = TabularCPD(
    variable='Grade', variable_card=3,
    values=[
        [0.3, 0.7, 0.2, 0.5, 0.1, 0.4, 0.05, 0.2],  # A
        [0.4, 0.2, 0.5, 0.3, 0.3, 0.4, 0.25, 0.3],  # B
        [0.3, 0.1, 0.3, 0.2, 0.6, 0.2, 0.7, 0.5]   # C
    ],
    evidence=['Intelligence', 'StudyHours', 'Difficulty'],
    evidence_card=[2, 2, 2]
)

cpd_pass = TabularCPD(
    variable='Pass', variable_card=2,
    values=[
        [0.95, 0.8, 0.5],
        [0.05, 0.2, 0.5]
    ],
    evidence_card=[3],
    evidence=['Grade']
)

model.add_cpds(cpd_intelligence, cpd_studyHours, cpd_difficulty, cpd_grade, cpd_pass)

assert model.check_model(), "Model is incorrect!"

inference = VariableElimination(model)

result1 = inference.query(variables=['Pass'], evidence={'StudyHours': 0, 'Difficulty': 0})

print("P(Pass | StudyHours = Suff, Diff=Hard: ", result1)

P(Pass | StudyHours = Suff, Diff=Hard:  +---------+-------------+
| Pass    |   phi(Pass) |
+=========+=============+
| Pass(0) |      0.7190 |
+---------+-------------+
| Pass(1) |      0.2810 |
+---------+-------------+


In [26]:
# Task 4: MARKOV MODEL

import numpy as np

states = ["Sunny", "Cloudy", "Rainy"]   # All possible states

# --- Define transition matrix ---
transition_matrix = np.array([
    # To:  Sunny  Cloudy  Rainy
    [0.7,   0.2,   0.1],   # From Sunny
    [0.3,   0.4,   0.3],   # From Cloudy
    [0.2,   0.3,   0.5]    # From Rainy
])

def simulate_markov(initial_state, num_steps, states, transition_matrix):
    current_state = initial_state       # Start here
    sequence = [current_state]          # Record each state we visit

    for _ in range(num_steps):
        # Find the row index of current state in the states list
        state_index = states.index(current_state)

        # Pick next state using that row's probabilities
        next_state = np.random.choice(states, p=transition_matrix[state_index])

        sequence.append(next_state)     # Record new state
        current_state = next_state      # Move to new state

    return sequence

# --- Run simulation ---
initial_state = "Sunny"    # Starting state
num_steps = 10             # for next 10 days

sequence = simulate_markov(initial_state, num_steps, states, transition_matrix)

print(f"Weather sequence for {num_steps} days starting from {initial_state}:")
print(" -> ".join(sequence))

rainy_days = sequence.count("Rainy")   # Count how many Rainy states appeared
print(f"Number of Rainy days: {rainy_days} out of {num_steps} steps")

# --- Calculate probability of at least 3 rainy days ---
big_trials = 10000      # Run simulation many times to estimate probability
target = "Rainy"
min_count = 3           

success = 0
for _ in range(big_trials):
    seq = simulate_markov(initial_state, num_steps, states, transition_matrix)
    if seq.count(target) >= min_count:
        success += 1

prob = success / big_trials
print(f"\nP(at least {min_count} {target} days in {num_steps} steps): {prob:.3f}")

Weather sequence for 10 days starting from Sunny:
Sunny -> Sunny -> Sunny -> Sunny -> Sunny -> Sunny -> Sunny -> Sunny -> Sunny -> Sunny -> Sunny
Number of Rainy days: 0 out of 10 steps

P(at least 3 Rainy days in 10 steps): 0.403
